# HACK AI / Intro to Large Language Modelling
## Check for keywords in scraped web text and extract text context window

### *Mariam Cook*

### *m.cook6@exeter.ac.uk*

### *University of Exeter Centre for Computational Social Science*


In [ ]:
import pandas as pd

# Check for keywords

In [ ]:
keywordlisttocheck = ['Best Start in Life','Sure Start','family hubs', 'free school meals', 'School meals', 'Development checks',
                      'School readiness', 'breakfast clubs', 'free breakfast', 'childcare hours', 'free childcare hours',
                      'free childcare for working parents', 'tax-free childcare', 'tax free childcare', 'universal credit childcare']

In [ ]:
import re

def find_keywords_present(texttocheck, keywordlisttocheck):
  total_len = len(texttocheck)
  keywords_in_text = []
  full_results = []

  if(type(texttocheck) == str):
    texttocheck_lower = texttocheck.lower()

    for a_keyword in keywordlisttocheck:
      a_keyword_lower = a_keyword.lower()

      # Find all occurrences of the keyword
      start_pos = 0
      mention_count = 0

      while True:
        index = texttocheck_lower.find(a_keyword_lower, start_pos)

        if index == -1:  # No more occurrences found
          break

        mention_count += 1
        print(f'found this here (mention #{mention_count}): {a_keyword} at position {index}')

        keywords_in_text.append(a_keyword)

        # Calculate context window
        if index < 256:
          start_index = 0
        else:
          start_index = index - 256

        if index + 256 < total_len:
          to_end = 256
        else:
          to_end = total_len - index

        full_result = (a_keyword, mention_count, texttocheck_lower[start_index:index + to_end])
        if len(full_result[2]) > 0:
          full_results.append(full_result)

        # Move search position past this occurrence
        start_pos = index + len(a_keyword_lower)

  return full_results

In [ ]:
find_keywords_present('my example ---- test my result is Sure Start and here is some more text test some more and here is a giraffe test\
 my result is Sure Start and here is some more text test some more and here is a giraffe',
                      keywordlisttocheck)

found this here (mention #1): Sure Start at position 34
found this here (mention #2): Sure Start at position 127


[('Sure Start',
  1,
  'my example ---- test my result is sure start and here is some more text test some more and here is a giraffe test my result is sure start and here is some more text test some more and here is a giraffe'),
 ('Sure Start',
  2,
  'my example ---- test my result is sure start and here is some more text test some more and here is a giraffe test my result is sure start and here is some more text test some more and here is a giraffe')]

In [ ]:
find_keywords_present('my result is Sure Start', keywordlisttocheck)

found this here (mention #1): Sure Start at position 13


[('Sure Start', 1, 'my result is sure start')]

# Load News data

Data credit:

*   This data is the result of querying the 2024-25 common crawl UK government subset provided at the 2025 Bristol Datathon. [See here for the full data set (very large).](https://github.com/eshasadia/G5-CommonCrawl/blob/main/news_filtered_2024_p1_1.csv)
*   The data was further restricted to news only by [Meng Lee in our team: Group 5](https://github.com/eshasadia/G5-CommonCrawl/blob/main/data/filtered_2024_p1.csv).


In [ ]:
news_results_df = pd.DataFrame(columns=['source', 'id', 'url', 'parent_url', 'cc_url', 'content_truncated',
                                                 'result', 'mention_number', 'result_text'])
news_results_df

,source,id,url,parent_url,cc_url,content_truncated,result,mention_number,result_text


In [ ]:
df_news_filtered_2024_p1 = pd.read_csv('/content/news_filtered_2024_p1.csv')

df_news_filtered_2024_p1

,url,parent_url,postcodes,cc_url,content
0,https://www.gov.uk/government/news/nato-claims...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,nato claims 'successful year' in afghanistan -...
1,https://www.gov.uk/government/news/new-british...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,new british ambassador to afghanistan visits n...
2,https://www.gov.uk/government/news/corporal-st...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,corporal stephen paul curley killed in afghani...
3,https://www.gov.uk/government/news/127-2012-1-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,127/2012 - £1.3 million for communities gettin...
4,https://www.gov.uk/government/news/commercial-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,commercial partner announced to help improve m...
...,...,...,...,...,...
1886,https://www.gov.uk/government/news/nick-clegg-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,nick clegg sets out plans to end child detenti...
1887,https://www.gov.uk/government/news/praise-for-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,praise for local community in nottingham - gov...
1888,https://www.gov.uk/government/news/hundreds-of...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,hundreds of new zero emission buses to connect...
1889,https://www.gov.uk/government/news/deputy-prim...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,deputy prime minister launches more 'city deal...


In [ ]:
number_results = 0
for id, row in df_news_filtered_2024_p1.iterrows():
  result = find_keywords_present(row['content'], keywordlisttocheck)

  if len(result) > 0:
    for item in result:
      number_results += 1
      data_row = {}
      print(item)

      data_row['source'] = 'df_selected_urls_2024_p1'
      data_row['id'] = id
      data_row['url'] = row['url']
      data_row['parent_url'] = row['parent_url']
      data_row['content_truncated'] = row['content'][0:100]
      data_row['cc_url'] = row['cc_url']
      data_row['result'] = item[0]  # keyword
      data_row['mention_number'] = item[1]  # mention count
      data_row['result_text'] = item[2]  # context text

      news_results_df = pd.concat([news_results_df, pd.DataFrame([data_row])], ignore_index=True)

print('number_results', number_results)

found this here (mention #1): Sure Start at position 5238
('Sure Start', 1, ' we are recruiting 4,200 health visitors and expanding programmes such as family nurse partnerships. through establishing the £2bn a year early intervention grant, we have signalled the importance of this agenda. we have committed to maintain a network of sure start children’s centres and to expand the free offer of childcare to disadvantaged two year olds, and put in place measures like the pupil premium. we are encouraging system reform through community budgets and payment by results. this package will e')
found this here (mention #1): free school meals at position 8086
found this here (mention #2): free school meals at position 8160
found this here (mention #1): School meals at position 8091
found this here (mention #2): School meals at position 8165
('free school meals', 1, 'e right for local children. international evidence shows that giving teachers and heads more freedom in the classroom helps to raise

In [ ]:
news_results_df

,source,id,url,parent_url,cc_url,content_truncated,result,mention_number,result_text
0,df_selected_urls_2024_p1,80,https://www.gov.uk/government/news/graham-alle...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,graham allen launches second report on early i...,Sure Start,1,"we are recruiting 4,200 health visitors and e..."
1,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,free school meals,1,e right for local children. international evid...
2,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,free school meals,2,ers and heads more freedom in the classroom he...
3,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,School meals,1,ht for local children. international evidence ...
4,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,School meals,2,nd heads more freedom in the classroom helps t...
...,...,...,...,...,...,...,...,...,...
77,df_selected_urls_2024_p1,1876,https://www.gov.uk/government/news/more-help-w...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,more help with childcare costs for working fam...,tax-free childcare,1,minister's office and the rt hon nick clegg\n...
78,df_selected_urls_2024_p1,1876,https://www.gov.uk/government/news/more-help-w...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,more help with childcare costs for working fam...,tax-free childcare,2,r nick clegg said:\ni want to help every famil...
79,df_selected_urls_2024_p1,1876,https://www.gov.uk/government/news/more-help-w...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,more help with childcare costs for working fam...,tax-free childcare,3,antly worrying about how to juggle their famil...
80,df_selected_urls_2024_p1,1884,https://www.gov.uk/government/news/pupils-who-...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,pupils who fall behind in english and maths to...,free school meals,1,ce shows that pupils who are behind in english...


In [ ]:
df_news_filtered_2024_p2 = pd.read_csv('/content/news_filtered_2024_p2.csv')

df_news_filtered_2024_p2

,url,parent_url,postcodes,cc_url,content
0,https://www.gov.uk/government/news/new-virtual...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,new virtual cyber school gives teens chance to...
1,https://www.gov.uk/government/news/blueprint-f...,www.gov.uk,['SW1P 4DF'],https://data.commoncrawl.org/crawl-data/CC-MAI...,blueprint for 100 multi-million pound town dea...
2,https://www.gov.uk/government/news/opening-doo...,www.gov.uk,['SW1H 9EX'],https://data.commoncrawl.org/crawl-data/CC-MAI...,opening doors for young disabled people to eng...
3,https://www.gov.uk/government/news/workplace-d...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,workplace dispute reforms proposed by governme...
4,https://www.gov.uk/government/news/home-secret...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,home secretary hails security agreement with e...
...,...,...,...,...,...
977,https://www.gov.uk/government/news/funding-for...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,funding for two patient safety research centre...
978,https://www.gov.uk/government/news/new-divorce...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,new divorce law to end the blame game - gov.uk...
979,https://www.gov.uk/government/news/second-pubs...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,second pubs code declaration ends today - gov....
980,https://www.gov.uk/government/news/meeting-of-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,meeting of the withdrawal agreement joint comm...


In [ ]:
number_results = 0
for id, row in df_news_filtered_2024_p2.iterrows():
  result = find_keywords_present(row['content'], keywordlisttocheck)

  if len(result) > 0:
    for item in result:
      number_results += 1
      data_row = {}
      print(item)

      data_row['source'] = 'df_selected_urls_2024_p2'
      data_row['id'] = id
      data_row['url'] = row['url']
      data_row['parent_url'] = row['parent_url']
      data_row['cc_url'] = row['cc_url']
      data_row['content_truncated'] = row['content'][0:100]
      data_row['result'] = item[0]  # keyword
      data_row['mention_number'] = item[1]  # mention count
      data_row['result_text'] = item[2]  # context text

      news_results_df = pd.concat([news_results_df, pd.DataFrame([data_row])], ignore_index=True)

print('number_results', number_results)

found this here (mention #1): family hubs at position 5015
found this here (mention #1): childcare hours at position 4827
found this here (mention #1): free childcare hours at position 4822
('family hubs', 1, 'o be paid to early years providers to deliver the government’s free childcare hours.\nnew investment of £302 million to fund new programmes to support parents, provide bespoke breast feeding services and parent-infant mental support, and funding to rollout family hubs across england.\n£639 million resource funding per annum by 2024-25 as part of the government’s commitment to end rough sleeping in england, an 85% cash increase compared to 2019-20. this brings total funding to £1.9 billion resource and £109 mi')
('childcare hours', 1, 'april. alcohol duties will be frozen across the board for the third year running saving consumers £3 billion.\nadditional investment of £170 million in 2024-25 to increase the hourly rate to be paid to early years providers to deliver the government

In [ ]:
# 67 in the second part

In [ ]:
df_news_filtered_2025 = pd.read_csv('/content/news_filtered_2025.csv')

df_news_filtered_2025

,url,parent_url,postcodes,cc_url,content
0,https://www.gov.uk/government/news/uk-sanction...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,uk sanctions target 30 corrupt political figur...
1,https://www.gov.uk/government/news/new-funding...,www.gov.uk,['SW1P 4DF'],https://data.commoncrawl.org/crawl-data/CC-MAI...,new funding to help universities tackle antise...
2,https://govdiff.njk.onl/update/2025-10-03T07:3...,govdiff.njk.onl/update/2025-10-03T07:35:00+01:...,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,brexit guidance change explorer\nchange of htt...
3,https://govdiff.njk.onl/update/2025-10-03T09:2...,govdiff.njk.onl/update/2025-10-03T09:28:00+01:...,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,brexit guidance change explorer\nchange of htt...
4,https://www.gov.uk/government/news/nhs-staff-r...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,nhs staff receive pay rise - gov.uk\ncookies o...
...,...,...,...,...,...
1073,https://www.gov.uk/government/news/chiefs-of-s...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,chiefs of staff stand by sdsr - gov.uk\ncookie...
1074,https://www.gov.uk/government/news/ofsted-repo...,www.gov.uk,['SW1H 9EX'],https://data.commoncrawl.org/crawl-data/CC-MAI...,ofsted reports to thurrock council following f...
1075,https://www.gov.uk/government/news/opening-of-...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,opening of business applications for the new u...
1076,https://www.gov.uk/government/news/self-employ...,www.gov.uk,[],https://data.commoncrawl.org/crawl-data/CC-MAI...,self-employed invited to get ready to make the...


In [ ]:
number_results = 0
for id, row in df_news_filtered_2025.iterrows():
  result = find_keywords_present(row['content'], keywordlisttocheck)

  if len(result) > 0:
    for item in result:
      number_results += 1
      data_row = {}
      print(item)

      data_row['source'] = 'df_news_filtered_2025'
      data_row['id'] = id
      data_row['url'] = row['url']
      data_row['parent_url'] = row['parent_url']
      data_row['cc_url'] = row['cc_url']
      data_row['content_truncated'] = row['content'][0:100]
      data_row['result'] = item[0]  # keyword
      data_row['mention_number'] = item[1]  # mention count
      data_row['result_text'] = item[2]  # context text

      news_results_df = pd.concat([news_results_df, pd.DataFrame([data_row])], ignore_index=True)

print('number_results', number_results)

found this here (mention #1): free school meals at position 5094
found this here (mention #2): free school meals at position 5866
found this here (mention #1): School meals at position 5099
found this here (mention #2): School meals at position 5871
('free school meals', 1, 'ations, including government departments, the devolved administrations, local authorities, and other bodies both within and outside of government, including commercial organisations. passported benefits can be differentiated into: benefits in kind such as free school meals, free milk, fruit and vegetables, and vitamins, free prescriptions; cash benefits such as support for travel costs to hospitals or prisons; and discounts on charges or fees such as leisure discounts.\ngovernment departments and the devolve')
('free school meals', 2, ' current criteria for defining eligibility to various passported benefits will no longer exist.\nfootnote\nby passported benefits we mean those benefits to which working-age claimant

In [ ]:
news_results_df

,source,id,url,parent_url,cc_url,content_truncated,result,mention_number,result_text
0,df_selected_urls_2024_p1,80,https://www.gov.uk/government/news/graham-alle...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,graham allen launches second report on early i...,Sure Start,1,"we are recruiting 4,200 health visitors and e..."
1,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,free school meals,1,e right for local children. international evid...
2,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,free school meals,2,ers and heads more freedom in the classroom he...
3,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,School meals,1,ht for local children. international evidence ...
4,df_selected_urls_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,School meals,2,nd heads more freedom in the classroom helps t...
...,...,...,...,...,...,...,...,...,...
159,df_news_filtered_2025,872,https://www.gov.uk/government/news/schools-col...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,"schools, colleges and early years settings to ...",free school meals,2,"or free school meals, schools will be able to ..."
160,df_news_filtered_2025,872,https://www.gov.uk/government/news/schools-col...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,"schools, colleges and early years settings to ...",School meals,1,e children i am confident we will help beat th...
161,df_news_filtered_2025,872,https://www.gov.uk/government/news/schools-col...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,"schools, colleges and early years settings to ...",School meals,2,"ee school meals, schools will be able to provi..."
162,df_news_filtered_2025,886,https://www.gov.uk/government/news/employment-...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,employment support launched for over a million...,free school meals,1,losed of individuals who are solely eligible f...


In [ ]:
news_results_df

,source,id,url,parent_url,cc_url,content_truncated,result,mention_number,result_text


In [ ]:
news_results_df

,source,id,url,parent_url,cc_url,content_truncated,result,result_text
0,df_news_filtered_2024_p1,80,https://www.gov.uk/government/news/graham-alle...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,graham allen launches second report on early i...,sure start,"we are recruiting 4,200 health visitors and e..."
1,df_news_filtered_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,free school meals,e right for local children. international evid...
2,df_news_filtered_2024_p1,261,https://www.gov.uk/government/news/first-speci...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,first special and alternative provision free s...,school meals,ht for local children. international evidence ...
3,df_news_filtered_2024_p1,403,https://www.gov.uk/government/news/prime-minis...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,prime minister announces support for parents a...,best start in life,videos showing midwives demonstrating practic...
4,df_news_filtered_2024_p1,412,https://www.gov.uk/government/news/coasting-sc...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,coasting schools meeting - gov.uk\ncookies on ...,best start in life,"ure, and turned that school around. now he’s g..."
...,...,...,...,...,...,...,...,...
68,df_news_filtered_2025,671,https://www.gov.uk/government/news/free-chicke...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,free chickenpox vaccination offered for first ...,school meals,y and prevent sickness from these highly conta...
69,df_news_filtered_2025,872,https://www.gov.uk/government/news/schools-col...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,"schools, colleges and early years settings to ...",free school meals,erable children i am confident we will help be...
70,df_news_filtered_2025,872,https://www.gov.uk/government/news/schools-col...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,"schools, colleges and early years settings to ...",school meals,e children i am confident we will help beat th...
71,df_news_filtered_2025,886,https://www.gov.uk/government/news/employment-...,www.gov.uk,https://data.commoncrawl.org/crawl-data/CC-MAI...,employment support launched for over a million...,free school meals,losed of individuals who are solely eligible f...


In [ ]:
# export csv results

news_results_df.to_csv('news_results_df_for_analysis.csv')